# House Price Prediction - Kaggle Competition
## Home Data for ML Course

This notebook walks through the complete process of predicting house prices using machine learning.

## Step 1: Import Required Libraries
First, we need to import all the libraries we'll use throughout this project.

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Display settings
pd.set_option('display.max_columns', None)

print("Libraries imported successfully!")

## Step 2: Load the Data
Load both training and test datasets from the downloaded CSV files.

In [ ]:
# Load training data
train_data = pd.read_csv('train.csv')

# Load test data
test_data = pd.read_csv('test.csv')

print(f"Training data shape: {train_data.shape}")
print(f"Test data shape: {test_data.shape}")

## Step 3: Explore the Training Data
Let's look at the first few rows and understand what data we have.

In [ ]:
# Display first 5 rows
train_data.head()

In [ ]:
# Check data types and missing values
train_data.info()

In [ ]:
# Statistical summary of numerical columns
train_data.describe()

## Step 4: Examine the Target Variable (SalePrice)
Let's look at the distribution of house prices.

In [ ]:
# Basic statistics of SalePrice
print("Sale Price Statistics:")
print(f"Mean: ${train_data['SalePrice'].mean():,.2f}")
print(f"Median: ${train_data['SalePrice'].median():,.2f}")
print(f"Min: ${train_data['SalePrice'].min():,.2f}")
print(f"Max: ${train_data['SalePrice'].max():,.2f}")

In [ ]:
# Visualize SalePrice distribution
plt.figure(figsize=(10, 6))
plt.hist(train_data['SalePrice'], bins=50, edgecolor='black')
plt.xlabel('Sale Price')
plt.ylabel('Frequency')
plt.title('Distribution of House Sale Prices')
plt.ticklabel_format(style='plain', axis='x')
plt.show()

## Step 5: Check for Missing Values
Identify columns with missing data.

In [ ]:
# Count missing values in each column
missing_values = train_data.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)

print("Columns with missing values:")
print(missing_values)

## Step 6: Select Features
Choose which columns (features) to use for prediction. We'll start with numeric columns that don't have missing values.

In [ ]:
# Select only numeric columns
numeric_cols = train_data.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remove 'Id' and 'SalePrice' from features
numeric_cols = [col for col in numeric_cols if col not in ['Id', 'SalePrice']]

print(f"Total numeric features available: {len(numeric_cols)}")
print("\nNumeric features:")
print(numeric_cols)

In [ ]:
# For simplicity, let's select features with NO missing values
features_no_missing = [col for col in numeric_cols if train_data[col].isnull().sum() == 0]

print(f"Features with no missing values: {len(features_no_missing)}")
print("\nSelected features:")
print(features_no_missing)

## Step 7: Prepare Training Data (X and y)
Separate features (X) and target variable (y).

In [ ]:
# Features (X)
X = train_data[features_no_missing]

# Target variable (y)
y = train_data['SalePrice']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

## Step 8: Split Data for Validation
Split the training data to validate our model's performance.

In [ ]:
# Split into training and validation sets (80-20 split)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, 
                                                        train_size=0.8, 
                                                        test_size=0.2,
                                                        random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Validation set size: {X_valid.shape[0]}")

## Step 9: Build and Train the Model
Create a Random Forest Regressor and train it on our data.

In [ ]:
# Create the model
model = RandomForestRegressor(n_estimators=100, 
                              random_state=42,
                              max_depth=10,
                              min_samples_split=5)

# Train the model
print("Training the model...")
model.fit(X_train, y_train)
print("Model training complete!")

## Step 10: Evaluate Model Performance
Check how well our model performs on the validation set.

In [ ]:
# Make predictions on validation set
predictions_valid = model.predict(X_valid)

# Calculate Mean Absolute Error
mae = mean_absolute_error(y_valid, predictions_valid)

print(f"Mean Absolute Error on Validation Set: ${mae:,.2f}")
print(f"Average House Price: ${y_valid.mean():,.2f}")
print(f"Error as % of Average Price: {(mae/y_valid.mean())*100:.2f}%")

In [ ]:
# Visualize predictions vs actual values
plt.figure(figsize=(10, 6))
plt.scatter(y_valid, predictions_valid, alpha=0.5)
plt.plot([y_valid.min(), y_valid.max()], [y_valid.min(), y_valid.max()], 'r--', lw=2)
plt.xlabel('Actual Sale Price')
plt.ylabel('Predicted Sale Price')
plt.title('Actual vs Predicted House Prices')
plt.ticklabel_format(style='plain')
plt.show()

## Step 11: Check Feature Importance
See which features are most important for predictions.

In [ ]:
# Get feature importance
feature_importance = pd.DataFrame({
    'feature': features_no_missing,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 Most Important Features:")
print(feature_importance.head(10))

In [ ]:
# Visualize top 15 features
plt.figure(figsize=(10, 8))
top_features = feature_importance.head(15)
plt.barh(top_features['feature'], top_features['importance'])
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Top 15 Most Important Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Step 12: Retrain on Full Training Data
Now train on ALL training data (not just the split) for final predictions.

In [ ]:
# Train on full training dataset
print("Retraining model on full training data...")
final_model = RandomForestRegressor(n_estimators=100, 
                                    random_state=42,
                                    max_depth=10,
                                    min_samples_split=5)

final_model.fit(X, y)
print("Final model training complete!")

## Step 13: Prepare Test Data
Select the same features from test data.

In [ ]:
# Check test data
print("Test data overview:")
print(test_data.head())
print(f"\nTest data shape: {test_data.shape}")

In [ ]:
# Select same features from test data
X_test = test_data[features_no_missing]

print(f"Test features shape: {X_test.shape}")
print("\nChecking for missing values in test data:")
print(X_test.isnull().sum().sum())

## Step 14: Make Predictions on Test Data
Use our trained model to predict house prices for the test set.

In [ ]:
# Make predictions
test_predictions = final_model.predict(X_test)

print(f"Number of predictions: {len(test_predictions)}")
print(f"\nFirst 10 predictions:")
print(test_predictions[:10])

In [ ]:
# Statistics of predictions
print("Prediction Statistics:")
print(f"Mean predicted price: ${test_predictions.mean():,.2f}")
print(f"Median predicted price: ${np.median(test_predictions):,.2f}")
print(f"Min predicted price: ${test_predictions.min():,.2f}")
print(f"Max predicted price: ${test_predictions.max():,.2f}")

## Step 15: Create Submission File
Format predictions for Kaggle submission.

In [ ]:
# Create submission dataframe
submission = pd.DataFrame({
    'Id': test_data['Id'],
    'SalePrice': test_predictions
})

print("Submission file preview:")
print(submission.head(10))
print(f"\nSubmission shape: {submission.shape}")

In [ ]:
# Save to CSV file
submission.to_csv('submission.csv', index=False)

print("✅ Submission file 'submission.csv' created successfully!")
print("\nYou can now upload this file to Kaggle!")

## Step 16: Summary and Next Steps

### What We Accomplished:
1. ✅ Loaded and explored the data
2. ✅ Selected relevant features
3. ✅ Trained a Random Forest model
4. ✅ Evaluated model performance
5. ✅ Made predictions on test data
6. ✅ Created submission file

### How to Improve:
- Handle missing values in more features
- Include categorical features (use encoding)
- Try different models (XGBoost, LightGBM)
- Perform feature engineering
- Tune hyperparameters
- Use cross-validation

### To Submit:
1. Go to the Kaggle competition page
2. Click "Submit Predictions"
3. Upload `submission.csv`
4. Check your score on the leaderboard!

In [ ]:
# Verify submission file exists
import os

if os.path.exists('submission.csv'):
    print("✅ submission.csv file is ready!")
    print(f"File size: {os.path.getsize('submission.csv')} bytes")
else:
    print("❌ Submission file not found!")